# Linear Regression — Car Price Prediction

**Flow:** Load → Clean → EDA → Split → Feature Preparation → Train → Validate → Tune → Final Test → Predict

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

data = "https://raw.githubusercontent.com/alexeygrigorev/mlbookcamp-code/master/chapter-02-car-price/data.csv"

df = pd.read_csv(data)

# Clean column names and string values
df.columns = df.columns.str.lower().str.replace(" ", "_")

strings = df.select_dtypes(include=["object"]).columns

for column in strings:
    df[column] = df[column].str.lower().str.replace(" ", "_")

print("Shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

plt.figure(figsize=(8, 4))
sns.histplot(df['msrp'], bins=50)
plt.title("MSRP Distribution")
plt.show()

# Target transformation: log(1 + y)
y = np.log1p(df["msrp"])

## Train / Validation / Test

Dataset shuffle করে:

$$
60\% \rightarrow \text{train},\qquad
20\% \rightarrow \text{validation},\qquad
20\% \rightarrow \text{test}
$$

Validation দিয়ে model/tuning সিদ্ধান্ত নেওয়া হবে। Test set শুধু final evaluation-এর জন্য।

In [ ]:
np.random.seed(2)

idx = np.arange(len(df))
np.random.shuffle(idx)

df_shuffled = df.iloc[idx].reset_index(drop=True)

n = len(df_shuffled)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

df_train = df_shuffled.iloc[:n_train].copy()
df_val = df_shuffled.iloc[n_train:n_train + n_val].copy()
df_test = df_shuffled.iloc[n_train + n_val:].copy()

y_train = np.log1p(df_train["msrp"].values)
y_val = np.log1p(df_val["msrp"].values)
y_test = np.log1p(df_test["msrp"].values)

print("train:", len(df_train), "validation:", len(df_val), "test:", len(df_test))

## Feature Preparation

Baseline features-এর সাথে useful engineered/categorical features যোগ করি:

- `age = 2017 - year`
- `number_of_doors` → 2/3/4 binary features
- selected categorical columns-এর top-5 categories → binary features

সব preprocessing একটাই `prepare_x()` function-এ রাখা হয়েছে, তাই train/validation/test-এ একই logic ব্যবহার হবে।

In [ ]:
numerical_columns = [
    "engine_hp",
    "engine_cylinders",
    "highway_mpg",
    "city_mpg",
    "popularity"
]

categorical_columns = [
    "make",
    "engine_fuel_type",
    "transmission_type",
    "driven_wheels",
    "market_category",
    "vehicle_size",
    "vehicle_style"
]


categories = {}

for single_cat_column in categorical_columns:
    categories[single_cat_column] = df_train[single_cat_column].value_counts().head().index.to_list()


def prepare_x(df_train):
    df = df_train.copy()
    features = numerical_columns.copy() 

    df["age"] = 2017 - df["year"]
    features.append("age")

    for door in [2, 3, 4]:
        feature = f"num_doors_{door}" 
        df[feature] = (df["number_of_doors"] == door).astype(int)
        features.append(feature) 

    for column, values in categories.items():
        for value in values:
            feature = f"{column}__{value}"
            df[feature] = (df[column] == value).astype(int)
            features.append(feature)

    return df[features].fillna(0).values.astype(float) 

## Linear Regression from Scratch

Prediction:

$$
\hat{y} = w_0 + x_1w_1 + x_2w_2 + \cdots + x_nw_n
$$

Bias column of `1`s যোগ করলে:

$$
\hat{y}=Xw
$$

Normal Equation:

$$
w=(X^\top X)^{-1}X^\top y
$$

RMSE:

$$
\mathrm{RMSE}
=
\sqrt{
\frac{1}{m}
\sum_{i=1}^{m}(y_i-\hat{y}_i)^2
}
$$

In [ ]:
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTy = X.T.dot(y)

    w = np.linalg.inv(XTX).dot(XTy)

    return w[0], w[1:]

def predict(X, w0, w):
    return w0 + X.dot(w)

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

## Train + Validate

এটাই normal end-to-end training flow। Feature preparation আগেই function-এ আছে, তাই একই code বারবার লেখার দরকার নেই।

In [ ]:
X_train = prepare_x(df_train)
X_val = prepare_x(df_val)

w0, w = train_linear_regression(X_train, y_train)

y_pred = predict(X_val, w0, w)

print("Validation RMSE:", rmse(y_val, y_pred))

## Regularization

অনেক feature-এর কারণে \(X^TX\) unstable হতে পারে। Regularization stability বাড়াতে সাহায্য করে।

$$
w=(X^\top X+rI)^{-1}X^\top y
$$

এখানে \(r\) হলো regularization strength।

In [ ]:
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTy = X.T.dot(y)

    w = np.linalg.inv(XTX).dot(XTy)

    return w[0], w[1:]

## Tune \(r\)

Validation RMSE দেখে best regularization strength নির্বাচন করি। Test set এখানে ব্যবহার করা যাবে না।

In [ ]:
X_train = prepare_x(df_train)
X_val = prepare_x(df_val)

for r in [0, 1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1, 10]:
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)
    y_pred = predict(X_val, w0, w)

    print(f"r={r:<7} RMSE={rmse(y_val, y_pred):.6f}")

## Final Model

Best \(r\) validation result থেকে নেওয়ার পর train + validation একসাথে নিয়ে final model train করি। এরপর test set-এ একবার final RMSE বের করি।

In [ ]:
r = 0.001

df_full_train = pd.concat([df_train, df_val]).reset_index(drop=True)
y_full_train = np.concatenate([y_train, y_val])

X_full_train = prepare_x(df_full_train)
X_test = prepare_x(df_test)

w0, w = train_linear_regression_reg(
    X_full_train,
    y_full_train,
    r=r
)

y_pred_test = predict(X_test, w0, w)

print("Final Test RMSE:", rmse(y_test, y_pred_test))

## Predict One Car

Model log-price predict করে। Original price পেতে:

$$
\hat{price}=\exp(\hat{y}_{log})-1
$$

In [ ]:
car = df_test.iloc[20].to_dict()

X_small = prepare_x(pd.DataFrame([car]))

pred_log = predict(X_small, w0, w)[0]
pred_price = np.expm1(pred_log)

actual_price = np.expm1(y_test[20])

print("Predicted price:", pred_price)
print("Actual price:   ", actual_price)

## Final Revision Sheet

$$
\boxed{\hat{y}=w_0+Xw}
$$

$$
\boxed{w=(X^\top X)^{-1}X^\top y}
$$

$$
\boxed{w=(X^\top X+rI)^{-1}X^\top y}
$$

$$
\boxed{
\mathrm{RMSE}
=
\sqrt{\frac{1}{m}\sum_{i=1}^{m}(y_i-\hat{y}_i)^2}
}
$$

$$
\boxed{y_{log}=\log(1+y)}
\qquad
\boxed{y=\exp(y_{log})-1}
$$

**Mental flow:**  
**Clean → Split → Prepare X/Y → Train → Validate → Tune → Retrain on Train+Validation → Test → Predict**